In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# pubmedqa
# PRED_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/eval/ft_pubmedqa_v23500_True_Llama-3.2-1B-Instruct_predictions_add_unique_prompt.jsonl"
# DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/dataset/pubmedqa_v2/validation_gpt41mini_3500/data-00000-of-00001.arrow"

# squad
PRED_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/eval/ft_squad_v23500_True_Llama-3.2-1B-Instruct_predictions_add_unique_prompt.jsonl"
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/dataset/squad_v2/validation_gpt41mini_3500/data-00000-of-00001.arrow"

# hotpotqa
# PRED_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/eval/ft_hotpotqa3500_True_Llama-3.2-1B-Instruct_predictions_add_unique_prompt.jsonl"
# DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/RAFT/dataset/hotpotqa/validation_gpt41mini_3500/data-00000-of-00001.arrow"

TOP_K_CHECK_list = [1,2,3,4,5]

In [ ]:
import json, re
from datasets import Dataset



_ws = re.compile(r"\s+")
def norm_min(s: str) -> str:
    if s is None:
        return ""
    return _ws.sub(" ", str(s).strip())

def oracle_hit(oracle: str, used_docs: list[str], top_k=None) -> bool:
    o = norm_min(oracle)
    if o == "":
        return False
    docs = used_docs if top_k is None else used_docs[:top_k]
    for d in docs:
        dn = norm_min(d)
        # oracle is contained in doc, or doc is contained in oracle
        if o in dn or dn in o:
            return True
    return False

ds = Dataset.from_file(DATASET_PATH)
oracle_by_id = {row["id"]: row.get("oracle_context", "") for row in ds}

for TOP_K_CHECK in TOP_K_CHECK_list:
  n = 0
  hit = 0
  miss_examples = []

  with open(PRED_PATH, "r", encoding="utf-8") as f:
      for line in f:
          rec = json.loads(line)
          ex_id = rec["id"]
          used_docs = rec.get("used_docs", [])
          oracle = oracle_by_id.get(ex_id, "")

          ok = oracle_hit(oracle, used_docs, top_k=TOP_K_CHECK)
          n += 1
          hit += int(ok)

          if not ok and len(miss_examples) < 5:
              miss_examples.append({
                  "id": ex_id,
                  "question": rec.get("question", ""),
                  "oracle_head": norm_min(oracle)[:200],
                  "doc0_head": norm_min(used_docs[0])[:200] if used_docs else "",
                  "num_used_docs": len(used_docs),
              })

  print(f"Hit rate = {hit}/{n} = {hit/n*100:.1f} (top_k={TOP_K_CHECK})")


Hit rate = 2368/3500 = 67.7 (top_k=1)
Hit rate = 2789/3500 = 79.7 (top_k=2)
Hit rate = 2958/3500 = 84.5 (top_k=3)
Hit rate = 3063/3500 = 87.5 (top_k=4)
Hit rate = 3131/3500 = 89.5 (top_k=5)
